In [ ]:
# -*- coding: utf-8 -*-

cup_handle_weekly.py — Weekly Cup-with-Handle scan (t05 + staleness + HANDLE SCORE)

**Run this one file weekly.** Everything downstream of pattern detection happens here.

Pipeline:
1. Freshness check on ticker CSVs
2. Execute cup_handle_active_scanner.ipynb (live setups + refresh fires cache)
3. Compute Q5 staleness threshold from historical fires
4. Re-price setups at t05 (0.5 x cup depth)
5. Drop stalest 20% (Q5 filter)
6. **HANDLE SCORE + SIZE  <-- NEW.** Score every surviving setup on handle quality
   (short / fresh / shallow) against the FROZEN historical distribution, assign a
   discrete size bucket. Skips are FLAGGED, not dropped.
7. Build watchlist + dedupe by ticker
8. Save watchlist CSV
9. Print Bloomberg-ready paste block

Handle score (frozen in handle_score_thresholds.json by handle_score_freeze.py):
  Validated: good handle = SHORT, FRESH, SHALLOW. Top score-quintile PF ~4.2 IS /
  4.4 OOS; bottom ~breakeven. Features: days_since_handle_low, handle_dur_days,
  handle_depth_atr (all lower-is-better).
  Size map:  Q5->full | Q4->full | Q3->half | Q2->skip | Q1->skip

Locked deployment spec (t05):
- Entry: next-day open after breakout
- Stop: handle_low x 0.999, floor 1x ATR, cap 10% of entry
- Target: breakout + 0.5 x cup_depth
- Time stop: 120 days
- Universe filter: stock 50-SMA > 200-SMA
- Staleness filter: drop setups in top 20% of days_since_handle_low

In [ ]:
# ---------------------------------------------------------------------------
# papermill PARAMETERS (this cell is tagged `parameters`)
#
# papermill injects an override cell immediately BELOW this one. The config cell
# further down derives OUT_DIR, FIRES_CSV, THRESHOLDS_JSON, WATCHLIST_CSV, ...
# from these, so the overrides must land ABOVE that derivation -- hence a
# dedicated params cell rather than tagging the config cell itself.
#
# Defaults reproduce the original Colab behaviour exactly.
# ---------------------------------------------------------------------------
DATA_DIR = '/content/drive/MyDrive/Bukowski'   # per-ticker <TICKER>.csv corpus
OUT_DIR = ''                                   # '' -> derived as DATA_DIR/results
SCANNER_ALREADY_RAN = False                    # True when the papermill chain ran it


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path
import glob, warnings, json
warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass

# DATA_DIR / OUT_DIR arrive from the `parameters` cell above. Coerced to Path --
# every constant below is built with the / operator, so the Path type matters.
DATA_DIR = Path(DATA_DIR)
OUT_DIR  = Path(OUT_DIR) if OUT_DIR else DATA_DIR / 'results'

SCANNER_NB     = DATA_DIR / 'cup_handle_active_scanner.ipynb'
FIRES_CSV      = OUT_DIR / 'cup_handle_fires.csv'
V2B_FIRES_CSV  = OUT_DIR / 'cup_handle_v2b_fires_with_clean_volume.csv'
T05_TRADES_CSV = OUT_DIR / 'cup_handle_v2b_exit_sweep' / 'trades_t05.csv'
PENDING_CSV    = OUT_DIR / 'cup_handle_active_setups.csv'
THRESHOLDS_JSON = OUT_DIR / 'handle_score_thresholds.json'   # <-- NEW: frozen handle-score

WATCHLIST_CSV  = OUT_DIR / 'cup_handle_t05_watchlist.csv'

RECENT_DAYS = 30
TARGET_MULT = 0.5     # t05
STALENESS_QUANTILE = 0.80   # drop Q5 = drop top 20%

print('Config loaded.')

# --------------------------------------------------------------------------- #
# HANDLE SCORE helpers (NEW)                                                   #
# --------------------------------------------------------------------------- #
def load_handle_thresholds(path=THRESHOLDS_JSON):
    """Load the frozen handle-score thresholds. Returns None if not present
    (scoring is then skipped gracefully and every setup is flagged 'unscored')."""
    if not Path(path).exists():
        print(f'⚠️  {Path(path).name} not found — run handle_score_freeze.py once. '
              f'Skipping handle score/sizing this run.')
        return None
    with open(path) as fh:
        TH = json.load(fh)
    print(f'Handle score loaded: {TH["version"]} '
          f'(built from {TH["built_from"]["n_trades"]} trades, PF {TH["built_from"]["overall_pf"]}). '
          f'Size map Q5/Q4=full, Q3=half, Q1/Q2=skip.')
    return TH


def _ensure_handle_features(df, feats):
    """Make sure the three handle-quality features exist on df. Derive the two
    geometry features from base columns when the scanner didn't emit them
    (recent-breakout rows come from the fires cache and carry raw columns)."""
    df = df.copy()
    if 'handle_dur_days' not in df.columns and {'handle_low_date', 'rch_date'}.issubset(df.columns):
        df['handle_dur_days'] = (pd.to_datetime(df['handle_low_date'])
                                 - pd.to_datetime(df['rch_date'])).dt.days
    if 'handle_depth_atr' not in df.columns and {'breakout_level', 'handle_low', 'atr'}.issubset(df.columns):
        df['handle_depth_atr'] = (df['breakout_level'] - df['handle_low']) / df['atr']
    return df


def score_and_size(df, TH):
    """Score each setup on handle quality vs the FROZEN historical distribution and
    assign a discrete size bucket. Lower feature = better handle, so score is the
    mean of (1 - empirical_percentile_vs_history) across the three features."""
    if df is None or len(df) == 0:
        return df
    if TH is None:
        df = df.copy(); df['handle_score'] = np.nan; df['size_bucket'] = 'unscored'
        return df
    feats = TH['features']
    df = _ensure_handle_features(df, feats)
    hist = {c: np.sort(np.asarray(TH['feature_hist'][c], float)) for c in feats}
    s = np.zeros(len(df)); n_ok = 0
    for c in feats:
        if c not in df.columns:
            continue
        vals = pd.to_numeric(df[c], errors='coerce').values
        pct = np.searchsorted(hist[c], vals, side='right') / max(len(hist[c]), 1)
        pct = np.where(np.isnan(vals), 0.5, pct)   # missing feature -> neutral
        s += (1.0 - pct); n_ok += 1
    df = df.copy()
    df['handle_score'] = s / max(n_ok, 1)
    edges = TH['hscore_edges']; smap = TH['size_map']

    def bucket(hs):
        k = int(np.searchsorted(edges, hs, side='right') - 1)
        k = min(max(k, 0), len(edges) - 2)          # clamp to 0..4
        return smap.get(str(k), 'half')
    df['size_bucket'] = df['handle_score'].apply(bucket)
    return df

## Step 1 — Freshness check

In [ ]:
spy = DATA_DIR / 'SPY.csv'
if spy.exists():
    df = pd.read_csv(spy, parse_dates=['Date'])
    last_date = df['Date'].max()
    ASOF_DATE = last_date
    days_stale = (pd.Timestamp.now() - last_date).days
    print(f'SPY.csv last date: {last_date.date()} ({days_stale} days ago)')
    if days_stale > 3:
        print('⚠️  Ticker CSVs may be stale — refresh price data before running.')
    else:
        print('✓ Ticker data looks current.')
else:
    print('⚠️  SPY.csv not found — skipping freshness check.')

## Step 2 — Run the scanner

Executes cup_handle_active_scanner.ipynb inline via %run. All variables from that
notebook become available here. Scanner writes cup_handle_active_setups.csv
(pending + just_fired, now incl. handle_dur_days & handle_depth_atr) and refreshes
cup_handle_fires.csv (historical breakouts). If the scanner errors, the weekly
warns and continues on the existing cache rather than dying.

In [ ]:
# Step 2 -- Run the scanner.
#
# The `%run` line magic does NOT survive papermill, so the VPS pipeline
# (pipeline/run_detector.py) executes cup_handle_active_scanner.ipynb as its own
# papermill run FIRST and then sets SCANNER_ALREADY_RAN=True here. The two
# notebooks are coupled only through the setups CSV on disk (the scanner writes
# OUT_DIR/cup_handle_active_setups.csv, Step 5 below reads it), never through
# shared Python state -- so running them as two processes is equivalent.
#
# In Colab SCANNER_ALREADY_RAN stays False and the original %run path is used,
# unchanged.
_scanner_ok = False

if SCANNER_ALREADY_RAN:
    print('Scanner already executed upstream (papermill chain) -- skipping %run.')
    _scanner_ok = True
else:
    try:
        _ip = get_ipython()
    except NameError:
        _ip = None

    if _ip is not None and SCANNER_NB.exists():
        try:
            print(f'Running scanner: {SCANNER_NB.name} ...')
            _ip.run_line_magic('run', f'"{SCANNER_NB}"')
            _scanner_ok = True
            print('\u2713 Scanner complete \u2014 live setups refreshed.')
        except Exception as _e:
            print(f'\u26a0\ufe0f  Scanner run FAILED: {_e}')
            print('    Continuing on existing cache \u2014 setups may be stale. Fix the scanner and re-run.')
    elif _ip is None:
        print('\u26a0\ufe0f  Not running under IPython/Colab \u2014 cannot %run the scanner notebook.')
        print('    Run cup_handle_active_scanner.ipynb manually first, then this file.')
    else:
        print(f'\u26a0\ufe0f  Scanner notebook not found at {SCANNER_NB}. Continuing on existing cache.')


## Step 3 — Compute Q5 staleness threshold

In [ ]:
hist = pd.read_csv(V2B_FIRES_CSV, parse_dates=['handle_low_date', 'entry_date'])
hist['days_since_handle_low'] = (hist['entry_date'] - hist['handle_low_date']).dt.days

Q5_THRESHOLD_DAYS = hist['days_since_handle_low'].quantile(STALENESS_QUANTILE)

print(f'Historical fires: {len(hist)}')
print(f'days_since_handle_low distribution:')
print(f'  min    {hist["days_since_handle_low"].min():.0f}')
print(f'  median {hist["days_since_handle_low"].median():.0f}')
print(f'  p80 (Q5 threshold) {Q5_THRESHOLD_DAYS:.0f}')
print(f'  max    {hist["days_since_handle_low"].max():.0f}')
print(f'\n>>> STALENESS FILTER: drop setups where days_since_handle_low > {Q5_THRESHOLD_DAYS:.0f} <<<')

# Load frozen handle-score thresholds up front (used in Step 6)
TH = load_handle_thresholds()

## Step 4 — Recent breakouts, re-priced at t05

In [ ]:
fires = pd.read_csv(FIRES_CSV, parse_dates=[
    'lch_date','cup_low_date','rch_date','handle_low_date',
    'fire_date','confirm_date','entry_date'
])

if 'stock_uptrend' in fires.columns:
    fires = fires[fires['stock_uptrend'] == True].copy()

today = fires['entry_date'].max()
recent = fires[fires['entry_date'] >= today - pd.Timedelta(days=RECENT_DAYS)].copy()
recent['days_since_entry'] = (today - recent['entry_date']).dt.days
recent['days_since_handle_low'] = (recent['entry_date'] - recent['handle_low_date']).dt.days

recent['cup_depth_dollars'] = recent['target_price'] - recent['breakout_level']
recent['t05_target'] = recent['breakout_level'] + TARGET_MULT * recent['cup_depth_dollars']
recent['risk'] = recent['entry_price'] - recent['stop_price']
recent['R_to_target'] = (recent['t05_target'] - recent['entry_price']) / recent['risk']

def latest_close(ticker):
    f = DATA_DIR / f'{ticker}.csv'
    if not f.exists():
        return None
    d = pd.read_csv(f, usecols=['Date','Close'])
    return float(d['Close'].iloc[-1])

recent['current_price'] = recent['ticker'].apply(latest_close)
recent = recent.dropna(subset=['current_price']).copy()

recent['t05_status'] = np.where(
    recent['current_price'] >= recent['t05_target'], 'target_hit',
    np.where(recent['current_price'] <= recent['stop_price'], 'stopped_out', 'open')
)
recent['days_to_time_stop'] = 168 - recent['days_since_entry']

still_open = recent[(recent['t05_status'] == 'open') & (recent['days_to_time_stop'] > 0)].copy()

print(f'Recent breakouts (last {RECENT_DAYS}d): {len(recent)}')
print(f'  Already hit t05 target: {(recent["t05_status"] == "target_hit").sum()}')
print(f'  Already stopped out:    {(recent["t05_status"] == "stopped_out").sum()}')
print(f'  Still open + tradeable: {len(still_open)}')

## Step 5 — Pending + just_fired, re-priced at t05

In [ ]:
fired = pd.DataFrame()
pending = pd.DataFrame()

if PENDING_CSV.exists():
    pdf = pd.read_csv(PENDING_CSV, parse_dates=['handle_low_date'])
    pdf['cup_depth_dollars'] = pdf['target'] - pdf['breakout_level']
    pdf['t05_target'] = pdf['breakout_level'] + TARGET_MULT * pdf['cup_depth_dollars']
    pdf['risk'] = pdf['entry_est'] - pdf['stop']
    pdf['R_to_target'] = (pdf['t05_target'] - pdf['entry_est']) / pdf['risk']
    pdf['days_since_handle_low'] = (ASOF_DATE - pdf['handle_low_date']).dt.days

    fired = pdf[pdf['status'] == 'just_fired'].copy()
    pending = pdf[pdf['status'] == 'pending'].copy()

    print(f'Just fired (broke out in last ≤2 days): {len(fired)}')
    print(f'Pending (handle formed, awaiting breakout): {len(pending)}')
else:
    print('⚠️  cup_handle_active_setups.csv not produced by scanner. Skipping pending/just_fired.')

## Step 6 — Apply staleness filter (drop Q5)

In [ ]:
def apply_staleness_filter(df, label):
    if len(df) == 0:
        return df
    n_before = len(df)
    kept = df[df['days_since_handle_low'] <= Q5_THRESHOLD_DAYS].copy()
    dropped = n_before - len(kept)
    print(f'  {label}: {n_before} → {len(kept)} (dropped {dropped} stale)')
    return kept

print(f'Staleness filter (drop days_since_handle_low > {Q5_THRESHOLD_DAYS:.0f}):')
still_open = apply_staleness_filter(still_open, 'Recent breakouts')
fired      = apply_staleness_filter(fired,      'Just fired')
pending    = apply_staleness_filter(pending,    'Pending')

## Step 6.5 — HANDLE SCORE + SIZE (NEW)

Score every surviving setup on handle quality against the frozen historical
distribution and assign a discrete size bucket. Skips are FLAGGED (kept in the
watchlist with size_bucket='skip'), never silently dropped.

In [ ]:
still_open = score_and_size(still_open, TH)
fired      = score_and_size(fired,      TH)
pending    = score_and_size(pending,    TH)

if TH is not None:
    _all = pd.concat([d for d in [still_open, fired, pending] if d is not None and len(d)],
                     ignore_index=True) if any(len(d) for d in [still_open, fired, pending]) else pd.DataFrame()
    if len(_all):
        print('Handle score / size buckets across all surviving setups:')
        vc = _all['size_bucket'].value_counts()
        for b in ['full', 'half', 'skip', 'unscored']:
            if b in vc:
                print(f'  {b:8s}: {vc[b]}')

## Step 7 — Build watchlist + dedupe by ticker

In [ ]:
rows = []

def _hs(r):  return round(float(r['handle_score']), 3) if 'handle_score' in r and pd.notna(r['handle_score']) else None
def _sb(r):  return r['size_bucket'] if 'size_bucket' in r else 'unscored'

for _, r in fired.iterrows():
    rows.append(dict(bucket='just_fired', priority=1, ticker=r['ticker'], status='just_fired',
        signal_date=None, handle_low_date=r['handle_low_date'].date() if pd.notna(r['handle_low_date']) else None,
        days_since_handle_low=int(r['days_since_handle_low']),
        handle_score=_hs(r), size_bucket=_sb(r),
        current_price=r['current_price'], entry=r['entry_est'], stop=r['stop'],
        t05_target=r['t05_target'], R_to_target=r['R_to_target'],
        breakout_level=r['breakout_level'], cup_depth_pct=r['cup_depth_pct'],
        handle_retr_pct=r['handle_retr_pct']))

for _, r in still_open.iterrows():
    rows.append(dict(bucket='recent_breakout', priority=2, ticker=r['ticker'], status=r['t05_status'],
        signal_date=r['entry_date'].date(), handle_low_date=r['handle_low_date'].date(),
        days_since_handle_low=int(r['days_since_handle_low']),
        handle_score=_hs(r), size_bucket=_sb(r),
        current_price=r['current_price'], entry=r['entry_price'], stop=r['stop_price'],
        t05_target=r['t05_target'], R_to_target=r['R_to_target'],
        breakout_level=r['breakout_level'], cup_depth_pct=r['cup_depth_pct'],
        handle_retr_pct=r['handle_retr_pct']))

for _, r in pending.iterrows():
    rows.append(dict(bucket='pending', priority=3, ticker=r['ticker'], status='pending',
        signal_date=None, handle_low_date=r['handle_low_date'].date() if pd.notna(r['handle_low_date']) else None,
        days_since_handle_low=int(r['days_since_handle_low']),
        handle_score=_hs(r), size_bucket=_sb(r),
        current_price=r['current_price'], entry=r['entry_est'], stop=r['stop'],
        t05_target=r['t05_target'], R_to_target=r['R_to_target'],
        breakout_level=r['breakout_level'], cup_depth_pct=r['cup_depth_pct'],
        handle_retr_pct=r['handle_retr_pct']))

watchlist = pd.DataFrame(rows)
n_pre_dedupe = len(watchlist)

watchlist = watchlist.sort_values(['ticker', 'priority']).drop_duplicates('ticker', keep='first')
# sort: best handles first within each bucket (full > half > skip, then high score)
_size_rank = {'full': 0, 'half': 1, 'skip': 2, 'unscored': 3}
watchlist['_sz'] = watchlist['size_bucket'].map(_size_rank).fillna(3)
watchlist = watchlist.drop(columns=['priority']).sort_values(
    ['bucket', '_sz', 'days_since_handle_low']).drop(columns=['_sz']).reset_index(drop=True)

print(f'Total setups before dedupe: {n_pre_dedupe}')
print(f'After dedupe by ticker:     {len(watchlist)}')
print(f'\nBy bucket:')
print(watchlist['bucket'].value_counts().to_string())
if 'size_bucket' in watchlist.columns:
    print(f'\nBy size bucket:')
    print(watchlist['size_bucket'].value_counts().to_string())

In [ ]:
# --- ENRICHMENT: sector + tier + priority (before Step 8 save) ---
import json, numpy as np
from pathlib import Path
SECTORS_JSON = Path(OUT_DIR) / 'ticker_sectors.json'
sector_map = json.load(open(SECTORS_JSON)) if SECTORS_JSON.exists() else {}
watchlist['sector'] = watchlist['ticker'].map(sector_map).fillna('Unknown')

# tier (Q1..Q5) from handle_score via the frozen edges
if TH is not None:
    edges = TH['hscore_edges']                       # [.036,.333,.456,.548,.657,.889]
    watchlist['tier'] = watchlist['handle_score'].apply(
        lambda s: f"Q{int(np.digitize(s, edges[1:-1]))+1}" if pd.notna(s) else None)
else:
    watchlist['tier'] = None

# priority = 50/50 blend of handle-score rank + R:R rank, over TRADEABLE setups only
watchlist['priority'] = np.nan
_m = watchlist['size_bucket'].isin(['full', 'half'])
if _m.any():
    _hs = watchlist.loc[_m, 'handle_score'].rank(pct=True)
    _rr = watchlist.loc[_m, 'R_to_target'].rank(pct=True)
    watchlist.loc[_m, 'priority'] = (0.5*_hs + 0.5*_rr).round(3)
watchlist = watchlist.sort_values('priority', ascending=False, na_position='last').reset_index(drop=True)

print("tiers:", watchlist['tier'].value_counts(dropna=False).to_dict())
print("sectors:", watchlist['sector'].value_counts().to_dict())
print("priority set on", int(watchlist['priority'].notna().sum()), "tradeable setups")

## Step 8 — Save watchlist

In [ ]:
watchlist.to_csv(WATCHLIST_CSV, index=False)
print(f'Saved: {WATCHLIST_CSV}')

print('\n' + '='*140)
print('  FINAL WATCHLIST  (size_bucket: full=Q5/Q4, half=Q3, skip=Q1/Q2)')
print('='*140)
with pd.option_context('display.max_rows', 100, 'display.width', None):
    print(watchlist.to_string(index=False, float_format=lambda x: f'{x:.2f}'))

## Step 9 — Bloomberg-ready paste block

In [ ]:
bloomberg_cols = ['ticker', 'bucket', 'status', 'size_bucket', 'tier', 'sector', 'priority',
                  'handle_score', 'handle_low_date', 'current_price', 'entry', 'stop', 't05_target','breakout_level',
                  'R_to_target', 'cup_depth_pct', 'handle_retr_pct', 'days_since_handle_low']
bloomberg_cols = [c for c in bloomberg_cols if c in watchlist.columns]

print('```csv')
print(watchlist[bloomberg_cols].to_csv(index=False), end='')
print('```')

# ===== TIMESTAMPED ARCHIVE — append at bottom, run every week =====
from datetime import datetime
from pathlib import Path

_ARCHIVE_DIR = Path(OUT_DIR) / 'watchlist_archive'
_ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

_stamp = datetime.now().strftime('%Y-%m-%d_%H%M')
_archive_path = _ARCHIVE_DIR / f'cup_handle_t05_watchlist_{_stamp}.csv'
watchlist.to_csv(_archive_path, index=False)
_existing = sorted(_ARCHIVE_DIR.glob('cup_handle_t05_watchlist_*.csv'))
print(f'Archived: {_archive_path.name}  ·  total on record: {len(_existing)}')

# ===== LIVE FIRE LOG — durable, screened, deduped record of weekly just-fired setups =====
from pathlib import Path
from datetime import datetime

LIVE_FIRES_CSV = Path(OUT_DIR) / 'cup_handle_live_fires.csv'

_new = watchlist[watchlist['bucket'] == 'just_fired'].copy()

if len(_new) == 0:
    print('No just-fired setups this run — live fire log unchanged.')
else:
    _run_date = datetime.now().strftime('%Y-%m-%d')
    _new.insert(0, 'logged_date', _run_date)

    if LIVE_FIRES_CSV.exists():
        _log = pd.read_csv(LIVE_FIRES_CSV)
        _combined = pd.concat([_log, _new], ignore_index=True)
    else:
        _combined = _new

    _combined['handle_low_date'] = _combined['handle_low_date'].astype(str)
    before = len(_combined)
    _combined = _combined.drop_duplicates(['ticker', 'handle_low_date'], keep='first').reset_index(drop=True)
    deduped = before - len(_combined)

    _combined.to_csv(LIVE_FIRES_CSV, index=False)
    print(f'Live fire log: +{len(_new)} this run, {deduped} dupes skipped, {len(_combined)} total.')
    print(f'  → {LIVE_FIRES_CSV}')
    print(f'\nThis run\'s new fires:')
    _cols = [c for c in ['ticker','size_bucket','handle_score','handle_low_date','entry','stop','t05_target','R_to_target'] if c in _new.columns]
    print(_new[_cols].to_string(index=False))